# Case RJ - Grupo 2

**Notebook para Realização da Modelagem da Camada Gold da Arquitetura Medalhão**

Nomes dos Integrantes:
- Anna Cristina
- Gabriel Guimarães
- Juan Perri
- Milton Salgado
- Richard
- Yuri

## Sobre este notebook

Lemos as 9 tabelas que o `Bronze_To_Silver.ipynb` já gravou e montamos o modelo da camada gold: um
esquema em estrela com uma fato (`fato_atendimento_gold`, uma linha por chamado), sete dimensões
(`dim_calendario`, `dim_hora`, `dim_faixa_espera`, `dim_canal`, `dim_motivo`, `dim_atendente`,
`dim_cliente`) e uma tabela de auditoria (`qualidade_chamado_gold`). Em cima disso, 14 tabelas
analíticas prontas pra BI/apresentação (visão geral, KPI mensal, comparativo ano a ano, funil, Pareto
de motivos, desempenho de atendentes, drivers de satisfação, simulação de ganhos, entre outras).

- **Idempotente**: toda tabela é recriada por inteiro (`overwrite`/`CREATE OR REPLACE`) — rodar de
  novo dá o mesmo resultado.
- **Nenhum chamado se perde**: todos os joins pra montar a fato são `LEFT`; o que não casa com nenhuma
  dimensão vira um membro desconhecido (chave negativa: `-1`, `-2` ou `-3`), nunca uma linha descartada.
- **Privacidade**: o `id_cliente` (CPF) entra na gold só como hash (`sha2`, sem segredo — decisão do
  grupo, já que o id do case é dado fictício, não CPF real de ninguém). `nome_atendente` não é dado
  restrito pelo projeto (a regra vale só pra CPF/nome/e-mail de cliente) e aparece normalmente.
- **Rodar por último**: depende das 9 tabelas do `Bronze_To_Silver.ipynb` já estarem gravadas.

## Configuração

In [0]:
from pyspark.sql import functions as F, Window
 
CATALOGO = "dados_vcredit"
SCHEMA = "default"
 
DESTINO_CLIENTES = f"{CATALOGO}.{SCHEMA}.clientes_silver"
DESTINO_ATENDENTES = f"{CATALOGO}.{SCHEMA}.atendentes_silver"
DESTINO_MOTIVOS = f"{CATALOGO}.{SCHEMA}.motivos_silver"
DESTINO_CANAIS = f"{CATALOGO}.{SCHEMA}.canais_silver"
DESTINO_DEPARA = f"{CATALOGO}.{SCHEMA}.canais_depara_silver"
DESTINO_CUSTOS = f"{CATALOGO}.{SCHEMA}.custos_silver"
DESTINO_PESQUISA = f"{CATALOGO}.{SCHEMA}.pesquisa_satisfacao_silver"
DESTINO_CHAMADOS_HORA = f"{CATALOGO}.{SCHEMA}.chamados_hora_silver"
DESTINO_CHAMADOS = f"{CATALOGO}.{SCHEMA}.chamados_silver"
 
GOLD_DIM_CALENDARIO = f"{CATALOGO}.{SCHEMA}.dim_calendario_gold"
GOLD_DIM_HORA = f"{CATALOGO}.{SCHEMA}.dim_hora_gold"
GOLD_DIM_FAIXA_ESPERA = f"{CATALOGO}.{SCHEMA}.dim_faixa_espera_gold"
GOLD_DIM_CANAL = f"{CATALOGO}.{SCHEMA}.dim_canal_gold"
GOLD_DIM_MOTIVO = f"{CATALOGO}.{SCHEMA}.dim_motivo_gold"
GOLD_DIM_ATENDENTE = f"{CATALOGO}.{SCHEMA}.dim_atendente_gold"
GOLD_DIM_CLIENTE = f"{CATALOGO}.{SCHEMA}.dim_cliente_gold"
GOLD_FATO = f"{CATALOGO}.{SCHEMA}.fato_atendimento_gold"
GOLD_QUALIDADE_CHAMADO = f"{CATALOGO}.{SCHEMA}.qualidade_chamado_gold"
 
GOLD_VISAO_GERAL = f"{CATALOGO}.{SCHEMA}.visao_geral_gold"
GOLD_KPI_MENSAL = f"{CATALOGO}.{SCHEMA}.kpi_mensal_gold"
GOLD_COMPARATIVO_YOY = f"{CATALOGO}.{SCHEMA}.comparativo_yoy_gold"
GOLD_FUNIL = f"{CATALOGO}.{SCHEMA}.funil_atendimento_gold"
GOLD_ETAPA_MENSAL = f"{CATALOGO}.{SCHEMA}.etapa_mensal_gold"
GOLD_MOTIVOS_PARETO = f"{CATALOGO}.{SCHEMA}.motivos_pareto_gold"
GOLD_MOTIVO_ETAPA = f"{CATALOGO}.{SCHEMA}.motivo_etapa_gold"
GOLD_DEMANDA_DIA_HORA = f"{CATALOGO}.{SCHEMA}.demanda_dia_hora_gold"
GOLD_ATENDENTES_DESEMPENHO = f"{CATALOGO}.{SCHEMA}.atendentes_desempenho_gold"
GOLD_SATISFACAO_DRIVERS = f"{CATALOGO}.{SCHEMA}.satisfacao_drivers_gold"
GOLD_PERFIL_CLIENTE = f"{CATALOGO}.{SCHEMA}.perfil_cliente_gold"
GOLD_QUALIDADE_DADOS = f"{CATALOGO}.{SCHEMA}.qualidade_dados_gold"
GOLD_IMPACTO_INDICADORES = f"{CATALOGO}.{SCHEMA}.impacto_indicadores_gold"
GOLD_SIMULACAO_GANHOS = f"{CATALOGO}.{SCHEMA}.simulacao_ganhos_gold"
 
LIMITE_REINCIDENCIA_DIAS = 7
NOTA_SATISFEITO_MIN = 4
AMOSTRA_MIN_ATENDENTE = 30
PCT_DEFLEXAO = 0.20
PCT_REDUCAO_REINCIDENCIA = 0.30
CUSTO_REFERENCIA_ENTREVISTA = 1_400_000.00
 
FAIXAS_ESPERA = [
    (1, "1) até 5 min", 300),
    (2, "2) 5 a 15 min", 900),
    (3, "3) 15 a 30 min", 1800),
    (4, "4) 30 a 60 min", 3600),
    (5, "5) mais de 60 min", None),
]
NOMES_MES = ["Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
             "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"]
NOMES_DIA = ["Domingo", "Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado"]

## Funções auxiliares

In [0]:
def limpar(c):
    texto = F.trim(F.regexp_replace(c.cast("string"), r"[\s\u00a0]+", " "))
    return F.when(~F.lower(texto).isin("", "null", "none", "nan"), texto)
 
def norm_key(c):
    texto = F.translate(
        F.lower(limpar(c)),
        "áàâãäéèêëíìîïóòôõöúùûüçñ",
        "aaaaaeeeeiiiiooooouuuucn"
    )
    texto = F.regexp_replace(texto, r"[_\-]+", " ")
    return F.trim(F.regexp_replace(texto, r"\s+", " "))
 
def com_desconhecidos(df, membros):
    for valores in membros:
        linha = spark.range(1).select(
            *[F.lit(valores.get(c.name)).cast(c.dataType).alias(c.name) for c in df.schema.fields]
        )
        df = df.unionByName(linha)
    return df
 
def gravar_gold(df, destino):
    (
        df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(destino)
    )
    print(f"{destino}: {spark.table(destino).count():,} linhas")
 
def criar_gold(destino, sql):
    spark.sql(f"CREATE OR REPLACE TABLE {destino} AS {sql}")
    print(f"{destino}: {spark.table(destino).count():,} linhas")

## Leitura de chamados

In [0]:
ch_raw = spark.table(DESTINO_CHAMADOS)
colunas_ch = [
    "id_chamado", "id_cliente", "canal_chave", "motivo_chave", "id_atendente", "resolvido",
    "dq_resolvido_invalido", "dq_resolvido_ausente", "dq_atendente_invalido", "status_atendente",
]
faltando = set(colunas_ch) - set(ch_raw.columns)
if faltando:
    raise ValueError(f"Silver de chamados sem as colunas: {sorted(faltando)}")
 
ch = ch_raw.select(
    "id_chamado",
    # sha2 puro (sem HMAC/segredo): decisao do grupo - dados ficticios do case, nao CPF real.
    F.sha2(F.col("id_cliente"), 256).alias("id_cliente_hash"),
    F.col("id_cliente").isNull().alias("sem_id_cliente"),
    "canal_chave",
    "motivo_chave",
    F.col("id_atendente").try_cast("bigint").alias("id_atendente_num"),
    "status_atendente",
    "resolvido",
    "dq_resolvido_invalido",
    "dq_resolvido_ausente",
    "dq_atendente_invalido",
)
 
n_chamados = ch.count()
if n_chamados == 0:
    raise ValueError("Silver de chamados vazia.")
if ch.where(F.col("id_chamado").isNull()).count() or ch.select("id_chamado").distinct().count() != n_chamados:
    raise ValueError("id_chamado ausente ou duplicado na Silver de chamados.")
 
print(f"chamados: {n_chamados:,}")

chamados: 131,838


## Leitura de tempos, custas e pesquisa

In [0]:
hr = spark.table(DESTINO_CHAMADOS_HORA).select(
    "id_chamado", "abandonado",
    "abertura_original_ts", "abertura_ts", "inicio_ts", "fim_original_ts", "fim_ts",
    "tempo_espera_seg", "tempo_atendimento_seg", "tempo_total_seg",
    "data_abertura", "hora_abertura",
    "ano_abertura_corrigido", "dia_fim_corrigido",
    "abertura_erro_parse", "inicio_erro_parse", "fim_erro_parse",
    F.lit(True).alias("tem_registro_hora"),
)
 
cu = (
    spark.table(DESTINO_CUSTOS)
    .withColumn("id_chamado", F.trim(F.col("id_chamado").cast("string")))
    .groupBy("id_chamado")
    .agg(
        F.sum(F.when(F.col("custo") > 0, F.col("custo"))).alias("custo"),
        F.count("*").alias("qtd_registros_custo"),
        (F.max(F.when(F.col("custo").isNull() | (F.col("custo") <= 0), 1).otherwise(0)) == 1).alias("custo_com_problema"),
    )
)
 
ps = spark.table(DESTINO_PESQUISA).select(
    F.trim(F.col("id_chamado").cast("string")).alias("id_chamado"),
    "nota", "nota_ajustada", "dq_nota_invalida",
    F.lit(True).alias("tem_pesquisa"),
)

## DIM | dim_calendario

In [0]:
limites = hr.agg(F.min("data_abertura").alias("ini"), F.max("data_abertura").alias("fim")).first()
if limites["ini"] is None:
    raise ValueError("Chamados Hora sem data de abertura válida.")
ini = limites["ini"].replace(month=1, day=1)
fim = limites["fim"].replace(month=12, day=31)
 
dim_calendario_base = (
    spark.sql(f"SELECT explode(sequence(DATE'{ini}', DATE'{fim}', INTERVAL 1 DAY)) AS data")
    .withColumn("id_data", F.date_format("data", "yyyyMMdd").cast("int"))
    .withColumn("ano", F.year("data"))
    .withColumn("semestre", F.when(F.month("data") <= 6, 1).otherwise(2))
    .withColumn("trimestre", F.quarter("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("nome_mes", F.element_at(F.array(*[F.lit(n) for n in NOMES_MES]), F.month("data")))
    .withColumn("ano_mes", F.date_format("data", "yyyy-MM"))
    .withColumn("dia", F.dayofmonth("data"))
    .withColumn("dia_semana", F.dayofweek("data"))
    .withColumn("nome_dia", F.element_at(F.array(*[F.lit(n) for n in NOMES_DIA]), F.dayofweek("data")))
    .withColumn("fim_de_semana", F.dayofweek("data").isin(1, 7))
    .select("id_data", "data", "ano", "semestre", "trimestre", "mes", "nome_mes",
            "ano_mes", "dia", "dia_semana", "nome_dia", "fim_de_semana")
)
 
dim_calendario = com_desconhecidos(
    dim_calendario_base,
    [{"id_data": -1, "nome_mes": "Sem data", "nome_dia": "Sem data"}],
)
gravar_gold(dim_calendario, GOLD_DIM_CALENDARIO)
 

dados_vcredit.default.dim_calendario_gold: 731 linhas


## DIM | dim_hora

In [0]:
dim_hora_base = spark.range(24).select(
    F.col("id").cast("int").alias("id_hora"),
    F.format_string("%02dh", F.col("id")).alias("rotulo_hora"),
    F.when(F.col("id") < 6, "Madrugada")
     .when(F.col("id") < 12, "Manhã")
     .when(F.col("id") < 18, "Tarde")
     .otherwise("Noite").alias("faixa_horaria"),
    F.when(F.col("id") < 6, 1)
     .when(F.col("id") < 12, 2)
     .when(F.col("id") < 18, 3)
     .otherwise(4).alias("ordem_faixa"),
)
 
dim_hora = com_desconhecidos(
    dim_hora_base,
    [{"id_hora": -1, "rotulo_hora": "Sem hora", "faixa_horaria": "Sem hora", "ordem_faixa": 5}],
)
gravar_gold(dim_hora, GOLD_DIM_HORA)

dados_vcredit.default.dim_hora_gold: 25 linhas


## DIM | dim_faixa_espera

In [0]:
dim_faixa_espera_base = spark.createDataFrame(
    FAIXAS_ESPERA,
    "id_faixa_espera int, rotulo_faixa string, limite_superior_seg int",
)
 
dim_faixa_espera = com_desconhecidos(
    dim_faixa_espera_base,
    [{"id_faixa_espera": -1, "rotulo_faixa": "Sem espera válida"}],
)
gravar_gold(dim_faixa_espera, GOLD_DIM_FAIXA_ESPERA)

dados_vcredit.default.dim_faixa_espera_gold: 6 linhas


## DIM | dim_canal

In [0]:
dim_canal_base = (
    spark.table(DESTINO_CANAIS)
    .select("id_canal", "nome_canal", "tipo_canal", "status_canal", "is_ativo")
    .withColumn(
        "etapa_funil",
        F.when(F.col("nome_canal").isin("URA", "CHATBOT"), "Autosserviço")
         .when(F.col("nome_canal") == "ATENDIMENTO INICIAL", "Nível 1")
         .when(F.col("nome_canal") == "ATENDIMENTO ESPECIALIZADO", "Nível 2")
         .otherwise("Não mapeado"),
    )
)
 
dim_canal = com_desconhecidos(
    dim_canal_base,
    [{"id_canal": -1, "nome_canal": "Não mapeado", "tipo_canal": "Não mapeado", "etapa_funil": "Não mapeado"}],
)
gravar_gold(dim_canal, GOLD_DIM_CANAL)

dados_vcredit.default.dim_canal_gold: 5 linhas


## DIM | dim_motivo

In [0]:
w_mo = Window.partitionBy("motivo_chave").orderBy("id_motivo")
dim_motivo_base = (
    spark.table(DESTINO_MOTIVOS)
    .withColumn("id_motivo", F.col("id_motivo").try_cast("bigint"))
    .withColumn("motivo_chave", norm_key(F.col("nome_motivo")))
    .where(F.col("id_motivo").isNotNull() & F.col("motivo_chave").isNotNull())
    .withColumn("_rn", F.row_number().over(w_mo))
    .where("_rn = 1")
    .select("id_motivo", "nome_motivo", "categoria", "criticidade", "motivo_chave")
)
 
n_motivos = dim_motivo_base.count()
if dim_motivo_base.select("id_motivo").distinct().count() != n_motivos:
    raise ValueError("id_motivo repetido para motivos diferentes na Silver de motivos.")
 
dim_motivo = com_desconhecidos(
    dim_motivo_base,
    [{"id_motivo": -1, "nome_motivo": "Não mapeado", "categoria": "Não mapeado", "criticidade": "não mapeado"}],
)
gravar_gold(dim_motivo, GOLD_DIM_MOTIVO)

dados_vcredit.default.dim_motivo_gold: 15 linhas


## DIM | dim_atendente

In [0]:
w_at = Window.partitionBy("id_atendente").orderBy(
    F.col("data_admissao").desc_nulls_last(), F.col("nivel_atendimento").asc_nulls_last()
)
dim_atendente_base = (
    spark.table(DESTINO_ATENDENTES)
    .withColumn("id_atendente", F.trim(F.col("id_atendente").cast("string")).try_cast("bigint"))
    .where(F.col("id_atendente").isNotNull())
    .withColumn("_rn", F.row_number().over(w_at))
    .where("_rn = 1")
    .select(
        "id_atendente",
        F.coalesce(
            F.when(F.length(F.trim(F.col("nome_atendente"))) > 0, F.trim(F.col("nome_atendente"))),
            F.concat(F.lit("Atendente "), F.col("id_atendente").cast("string")),
        ).alias("rotulo_atendente"),
        "nivel_atendimento",
        "data_admissao",
        "data_desligamento",
        "status_atendente",
        (F.col("status_atendente") == "ativo").alias("is_ativo"),
    )
)
 
dim_atendente = com_desconhecidos(
    dim_atendente_base,
    [
        {"id_atendente": -1, "rotulo_atendente": "Sem atendente informado"},
        {"id_atendente": -2, "rotulo_atendente": "Atendente inválido na origem"},
        {"id_atendente": -3, "rotulo_atendente": "Atendente fora do cadastro"},
    ],
)
gravar_gold(dim_atendente, GOLD_DIM_ATENDENTE)

dados_vcredit.default.dim_atendente_gold: 28 linhas


## DIM | dim_cliente

In [0]:
cli_raw = spark.table(DESTINO_CLIENTES)
 
if "id_cliente" in [c.lower() for c in cli_raw.columns]:
    idade = F.col("idade").cast("int")
    w_cl = Window.partitionBy("id_cliente_hash").orderBy("data_cadastro")
    dim_cliente = (
        cli_raw
        # sha2 puro (sem HMAC/segredo): mesma decisao da celula 5, dados ficticios do case.
        .withColumn("id_cliente_hash", F.sha2(limpar(F.col("id_cliente")), 256))
        .where(F.col("id_cliente_hash").isNotNull())
        .withColumn("_rn", F.row_number().over(w_cl))
        .where("_rn = 1")
        .select(
            "id_cliente_hash",
            F.when(idade.isNull(), "Sem idade")
             .when(idade < 25, "18-24")
             .when(idade < 35, "25-34")
             .when(idade < 45, "35-44")
             .when(idade < 55, "45-54")
             .when(idade < 65, "55-64")
             .otherwise("65+").alias("faixa_etaria"),
            F.coalesce(F.col("regiao"), F.lit("Sem região")).alias("regiao"),
            # convertidas pra date/int, como o guia descreve - a Silver entrega as duas como string.
            F.col("dia_vencimento_fatura").try_cast("int").alias("dia_vencimento_fatura"),
            F.to_date(F.col("data_cadastro"), "yyyy/MM/dd").alias("data_cadastro"),
        )
    )
    gravar_gold(dim_cliente, GOLD_DIM_CLIENTE)
    DIM_CLIENTE_CRIADA = True
else:
    spark.sql(f"DROP TABLE IF EXISTS {GOLD_DIM_CLIENTE}")
    DIM_CLIENTE_CRIADA = False
    print("dim_cliente_gold não criada: a Silver de clientes não tem id_cliente.")

dados_vcredit.default.dim_cliente_gold: 47,705 linhas


## ETL | Montagem da fato e da tabela qualidade

In [0]:
lk_canal = (
    spark.table(DESTINO_DEPARA).select("canal_chave", "id_canal")
    .join(dim_canal_base.select("id_canal", "etapa_funil"), "id_canal", "inner")
    .select("canal_chave", F.col("id_canal").alias("id_canal_fk"), "etapa_funil")
)
 
lk_motivo = dim_motivo_base.select("motivo_chave", F.col("id_motivo").alias("id_motivo_fk"))
 
lk_atendente = dim_atendente_base.select(
    F.col("id_atendente").alias("id_atendente_num"),
    F.col("nivel_atendimento").alias("nivel_atendente"),
    "data_admissao",
    "data_desligamento",
    F.lit(True).alias("atendente_encontrado"),
)
 
e = F.col("tempo_espera_seg")
faixa_expr = F.when(e.isNull(), F.lit(-1))
for id_faixa, _, limite in FAIXAS_ESPERA:
    if limite is None:
        faixa_expr = faixa_expr.otherwise(F.lit(id_faixa))
    else:
        faixa_expr = faixa_expr.when(e <= limite, F.lit(id_faixa))
 
w_reinc = Window.partitionBy("id_cliente_hash", "motivo_chave").orderBy("abertura_ts", "id_chamado")
 
base = (
    ch.join(hr, "id_chamado", "left")
      .join(cu, "id_chamado", "left")
      .join(ps, "id_chamado", "left")
      .join(F.broadcast(lk_canal), "canal_chave", "left")
      .join(F.broadcast(lk_motivo), "motivo_chave", "left")
      .join(F.broadcast(lk_atendente), "id_atendente_num", "left")
      .withColumn("tem_registro_hora", F.coalesce("tem_registro_hora", F.lit(False)))
      .withColumn("tem_pesquisa", F.coalesce("tem_pesquisa", F.lit(False)))
      .withColumn("atendente_encontrado", F.coalesce("atendente_encontrado", F.lit(False)))
      .withColumn("qtd_registros_custo", F.coalesce("qtd_registros_custo", F.lit(0)))
      .withColumn("custo_com_problema", F.coalesce("custo_com_problema", F.lit(False)))
      .withColumn("abertura_erro_parse", F.coalesce("abertura_erro_parse", F.lit(False)))
      .withColumn("inicio_erro_parse", F.coalesce("inicio_erro_parse", F.lit(False)))
      .withColumn("fim_erro_parse", F.coalesce("fim_erro_parse", F.lit(False)))
      .withColumn("ano_abertura_corrigido", F.coalesce("ano_abertura_corrigido", F.lit(False)))
      .withColumn("dia_fim_corrigido", F.coalesce("dia_fim_corrigido", F.lit(False)))
      .withColumn("canal_mapeado", F.col("id_canal_fk").isNotNull())
      .withColumn("motivo_mapeado", F.col("id_motivo_fk").isNotNull())
      .withColumn("id_data", F.coalesce(F.date_format("data_abertura", "yyyyMMdd").cast("int"), F.lit(-1)))
      .withColumn("id_hora", F.coalesce(F.col("hora_abertura"), F.lit(-1)))
      .withColumn("id_canal", F.coalesce(F.col("id_canal_fk"), F.lit(-1)))
      .withColumn("id_motivo", F.coalesce(F.col("id_motivo_fk"), F.lit(-1).cast("bigint")))
      .withColumn(
          "id_atendente",
          F.when(F.col("id_atendente_num").isNotNull() & F.col("atendente_encontrado"), F.col("id_atendente_num"))
           .when(F.col("id_atendente_num").isNotNull(), F.lit(-3))
           .when(F.col("status_atendente") == "invalido_na_origem", F.lit(-2))
           .otherwise(F.lit(-1)),
      )
      .withColumn("id_faixa_espera", faixa_expr)
      .withColumn("satisfeito", F.when(F.col("nota").isNotNull(), F.col("nota") >= NOTA_SATISFEITO_MIN))
      .withColumn("satisfeito_ajustado", F.when(F.col("nota_ajustada").isNotNull(), F.col("nota_ajustada") >= NOTA_SATISFEITO_MIN))
      .withColumn("atendente_nao_encontrado", F.col("id_atendente_num").isNotNull() & ~F.col("atendente_encontrado"))
      .withColumn(
          "atendente_fora_vigencia",
          F.coalesce(
              F.col("atendente_encontrado") & F.col("data_abertura").isNotNull() & (
                  (F.col("data_abertura") < F.col("data_admissao")) |
                  (F.col("data_desligamento").isNotNull() & (F.col("data_abertura") > F.col("data_desligamento")))
              ),
              F.lit(False),
          ),
      )
      .withColumn(
          "autosservico_com_atendente",
          F.coalesce((F.col("etapa_funil") == "Autosserviço") & F.col("id_atendente_num").isNotNull(), F.lit(False)),
      )
      .withColumn(
          "atendente_nivel_diverge_canal",
          F.coalesce(
              ((F.col("etapa_funil") == "Nível 1") & (F.col("nivel_atendente") == 2)) |
              ((F.col("etapa_funil") == "Nível 2") & (F.col("nivel_atendente") == 1)),
              F.lit(False),
          ),
      )
      .withColumn("_abertura_anterior", F.lag("abertura_ts").over(w_reinc))
      .withColumn(
          "reincidente_7d",
          F.coalesce(
              F.col("id_cliente_hash").isNotNull() & F.col("motivo_chave").isNotNull() &
              ((F.unix_timestamp("abertura_ts") - F.unix_timestamp("_abertura_anterior")) <= LIMITE_REINCIDENCIA_DIAS * 86400),
              F.lit(False),
          ),
      )
)
 
fato = base.select(
    "id_chamado", "id_cliente_hash",
    "id_data", "id_hora", "id_canal", "id_motivo", "id_atendente", "id_faixa_espera",
    "abertura_ts", "inicio_ts", "fim_ts",
    "tempo_espera_seg", "tempo_atendimento_seg", "tempo_total_seg",
    "custo", "nota", "nota_ajustada",
    "resolvido", "abandonado", "reincidente_7d", "satisfeito", "satisfeito_ajustado",
)
 
qualidade = base.select(
    "id_chamado",
    "tem_registro_hora", "abertura_erro_parse", "inicio_erro_parse", "fim_erro_parse",
    "ano_abertura_corrigido", "dia_fim_corrigido", "abertura_original_ts", "fim_original_ts",
    "canal_mapeado", "motivo_mapeado", "sem_id_cliente",
    "qtd_registros_custo", "custo_com_problema",
    "tem_pesquisa", "dq_nota_invalida",
    "dq_resolvido_invalido", "dq_resolvido_ausente", "dq_atendente_invalido", "status_atendente",
    "atendente_nao_encontrado", "atendente_fora_vigencia",
    "autosservico_com_atendente", "atendente_nivel_diverge_canal",
)

## ETL | Gravação da fato e da qualidade

In [0]:
gravar_gold(fato, GOLD_FATO)
gravar_gold(qualidade, GOLD_QUALIDADE_CHAMADO)
 
n_fato = spark.table(GOLD_FATO).count()
assert n_fato == n_chamados, f"Fato com {n_fato} linhas, mas há {n_chamados} chamados."

dados_vcredit.default.fato_atendimento_gold: 131,838 linhas
dados_vcredit.default.qualidade_chamado_gold: 131,838 linhas


## Verificação | Integridade referencial e chaves únicas

In [0]:
_fks = [
    ("id_data", GOLD_DIM_CALENDARIO),
    ("id_hora", GOLD_DIM_HORA),
    ("id_faixa_espera", GOLD_DIM_FAIXA_ESPERA),
    ("id_canal", GOLD_DIM_CANAL),
    ("id_motivo", GOLD_DIM_MOTIVO),
    ("id_atendente", GOLD_DIM_ATENDENTE),
]
 
_sql_orfaos = "\nUNION ALL\n".join(
    f"""SELECT '{fk}' AS chave, COUNT(*) AS chamados_sem_dimensao
FROM {GOLD_FATO} f LEFT JOIN {dim} d ON f.{fk} = d.{fk}
WHERE d.{fk} IS NULL"""
    for fk, dim in _fks
)
_sql_duplicadas = "\nUNION ALL\n".join(
    f"SELECT '{dim}' AS dimensao, COUNT(*) - COUNT(DISTINCT {fk}) AS chaves_duplicadas FROM {dim}"
    for fk, dim in _fks
)
 
_orfaos = spark.sql(_sql_orfaos).collect()
_duplicadas = spark.sql(_sql_duplicadas).collect()
assert all(r["chamados_sem_dimensao"] == 0 for r in _orfaos), _orfaos
assert all(r["chaves_duplicadas"] == 0 for r in _duplicadas), _duplicadas
 
display(spark.sql(_sql_orfaos))
display(spark.sql(_sql_duplicadas))

chave,chamados_sem_dimensao
id_data,0
id_hora,0
id_faixa_espera,0
id_canal,0
id_motivo,0
id_atendente,0


dimensao,chaves_duplicadas
dados_vcredit.default.dim_calendario_gold,0
dados_vcredit.default.dim_hora_gold,0
dados_vcredit.default.dim_faixa_espera_gold,0
dados_vcredit.default.dim_canal_gold,0
dados_vcredit.default.dim_motivo_gold,0
dados_vcredit.default.dim_atendente_gold,0


## Verificação | Chamados nos membros desconhecidos

In [0]:
display(spark.sql(f"""
SELECT 'canal' AS dimensao, COUNT_IF(id_canal = -1) AS chamados_desconhecidos, COUNT(*) AS total FROM {GOLD_FATO}
UNION ALL SELECT 'motivo', COUNT_IF(id_motivo = -1), COUNT(*) FROM {GOLD_FATO}
UNION ALL SELECT 'data', COUNT_IF(id_data = -1), COUNT(*) FROM {GOLD_FATO}
UNION ALL SELECT 'hora', COUNT_IF(id_hora = -1), COUNT(*) FROM {GOLD_FATO}
UNION ALL SELECT 'faixa_espera', COUNT_IF(id_faixa_espera = -1), COUNT(*) FROM {GOLD_FATO}
UNION ALL SELECT 'atendente: não informado', COUNT_IF(id_atendente = -1), COUNT(*) FROM {GOLD_FATO}
UNION ALL SELECT 'atendente: inválido na origem', COUNT_IF(id_atendente = -2), COUNT(*) FROM {GOLD_FATO}
UNION ALL SELECT 'atendente: fora do cadastro', COUNT_IF(id_atendente = -3), COUNT(*) FROM {GOLD_FATO}
"""))

dimensao,chamados_desconhecidos,total
canal,0,131838
motivo,0,131838
data,0,131838
hora,0,131838
faixa_espera,3414,131838
atendente: não informado,0,131838
atendente: inválido na origem,764,131838
atendente: fora do cadastro,0,131838


## Verificação | Valores sem correspondência

In [0]:
display(
    base.where(~F.col("motivo_mapeado")).select(F.lit("motivo").alias("campo"), F.col("motivo_chave").alias("valor"))
    .unionByName(base.where(~F.col("canal_mapeado")).select(F.lit("canal").alias("campo"), F.col("canal_chave").alias("valor")))
    .groupBy("campo", "valor").count()
    .orderBy(F.desc("count"))
    .limit(30)
)

campo,valor,count


## Consulta | visao_geral

In [0]:
criar_gold(GOLD_VISAO_GERAL, f"""
SELECT
  COUNT(*)                                                AS total_chamados,
  MIN(c.data)                                             AS primeira_data,
  MAX(c.data)                                             AS ultima_data,
  COUNT(DISTINCT c.ano_mes)                               AS meses_no_periodo,
  COUNT_IF(f.abandonado)                                  AS abandonados,
  ROUND(100 * AVG(CAST(f.abandonado AS INT)), 2)          AS pct_abandono,
  ROUND(100 * AVG(CAST(f.resolvido AS INT)), 2)           AS pct_resolucao,
  ROUND(AVG(f.tempo_espera_seg) / 60, 2)                  AS tme_min,
  ROUND(AVG(f.tempo_atendimento_seg) / 60, 2)             AS tma_min,
  ROUND(100 * AVG(CAST(f.satisfeito AS INT)), 2)          AS csat_pct,
  ROUND(AVG(f.nota), 2)                                   AS nota_media,
  ROUND(SUM(f.custo), 2)                                  AS custo_total,
  ROUND(AVG(f.custo), 2)                                  AS custo_medio_chamado,
  ROUND(SUM(f.custo) / COUNT(DISTINCT c.ano_mes), 2)      AS custo_mensal_medio,
  ROUND({CUSTO_REFERENCIA_ENTREVISTA}, 2)                 AS custo_referencia_entrevista,
  ROUND(SUM(f.custo) - {CUSTO_REFERENCIA_ENTREVISTA}, 2)  AS diferenca_vs_referencia
FROM {GOLD_FATO} f
JOIN {GOLD_DIM_CALENDARIO} c ON f.id_data = c.id_data
""")

dados_vcredit.default.visao_geral_gold: 1 linhas


## Consulta | kpi_mensal

In [0]:
criar_gold(GOLD_KPI_MENSAL, f"""
SELECT
  c.ano_mes,
  COUNT(*)                                                    AS chamados,
  COUNT_IF(f.abandonado)                                      AS abandonados,
  ROUND(100 * AVG(CAST(f.abandonado AS INT)), 2)              AS pct_abandono,
  ROUND(100 * AVG(CAST(f.resolvido AS INT)), 2)               AS pct_resolucao,
  ROUND(AVG(f.tempo_espera_seg) / 60, 2)                      AS tme_min,
  ROUND(percentile_approx(f.tempo_espera_seg, 0.9) / 60, 2)   AS p90_espera_min,
  ROUND(AVG(f.tempo_atendimento_seg) / 60, 2)                 AS tma_min,
  ROUND(SUM(f.custo), 2)                                      AS custo_total,
  ROUND(AVG(f.custo), 2)                                      AS custo_medio,
  ROUND(SUM(f.custo) / NULLIF(COUNT_IF(f.resolvido), 0), 2)   AS custo_por_resolucao,
  COUNT(f.nota)                                               AS respostas_pesquisa,
  ROUND(100 * COUNT(f.nota) / COUNT(*), 2)                    AS cobertura_pesquisa_pct,
  ROUND(100 * AVG(CAST(f.satisfeito AS INT)), 2)              AS csat_pct,
  ROUND(AVG(f.nota), 2)                                       AS nota_media,
  ROUND(100 * AVG(CAST(f.satisfeito_ajustado AS INT)), 2)     AS csat_ajustado_pct
FROM {GOLD_FATO} f
JOIN {GOLD_DIM_CALENDARIO} c ON f.id_data = c.id_data
WHERE c.ano_mes IS NOT NULL
GROUP BY c.ano_mes
ORDER BY c.ano_mes
""")

dados_vcredit.default.kpi_mensal_gold: 19 linhas


## Consulta | comparativo_yoy

In [0]:
def _yoy(expr, nome, casas=2):
    return ",\n  ".join(f"ROUND({expr.format(ano=a)}, {casas}) AS {nome}_{a}" for a in (2025, 2026))
 
_yoy_cols = ",\n  ".join([
    _yoy("COUNT_IF(c.ano = {ano})", "chamados", 0),
    _yoy("100 * AVG(CASE WHEN c.ano = {ano} THEN CAST(f.satisfeito AS INT) END)", "csat_pct"),
    _yoy("100 * AVG(CASE WHEN c.ano = {ano} THEN CAST(f.abandonado AS INT) END)", "pct_abandono"),
    _yoy("100 * AVG(CASE WHEN c.ano = {ano} THEN CAST(f.resolvido AS INT) END)", "pct_resolucao"),
    _yoy("AVG(CASE WHEN c.ano = {ano} THEN f.tempo_espera_seg END) / 60", "tme_min"),
    _yoy("AVG(CASE WHEN c.ano = {ano} THEN f.custo END)", "custo_medio"),
])
 
criar_gold(GOLD_COMPARATIVO_YOY, f"""
SELECT
  c.mes,
  c.nome_mes,
  {_yoy_cols}
FROM {GOLD_FATO} f
JOIN {GOLD_DIM_CALENDARIO} c ON f.id_data = c.id_data
WHERE c.ano IN (2025, 2026)
GROUP BY c.mes, c.nome_mes
ORDER BY c.mes
""")

dados_vcredit.default.comparativo_yoy_gold: 12 linhas


## Consulta | funil_atendimento

In [0]:
criar_gold(GOLD_FUNIL, f"""
SELECT
  ca.etapa_funil,
  ca.nome_canal                                               AS canal,
  ca.tipo_canal,
  COUNT(*)                                                    AS chamados,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)            AS pct_chamados,
  COUNT_IF(f.abandonado)                                      AS abandonados,
  ROUND(100 * AVG(CAST(f.abandonado AS INT)), 2)              AS pct_abandono,
  ROUND(100 * AVG(CAST(f.resolvido AS INT)), 2)               AS pct_resolucao,
  ROUND(AVG(f.tempo_espera_seg) / 60, 2)                      AS tme_min,
  ROUND(AVG(f.tempo_atendimento_seg) / 60, 2)                 AS tma_min,
  ROUND(SUM(f.custo), 2)                                      AS custo_total,
  ROUND(100 * SUM(f.custo) / SUM(SUM(f.custo)) OVER (), 2)    AS pct_custo,
  ROUND(AVG(f.custo), 2)                                      AS custo_medio,
  ROUND(SUM(f.custo) / NULLIF(COUNT_IF(f.resolvido), 0), 2)   AS custo_por_resolucao,
  ROUND(100 * AVG(CAST(f.satisfeito AS INT)), 2)              AS csat_pct
FROM {GOLD_FATO} f
JOIN {GOLD_DIM_CANAL} ca ON f.id_canal = ca.id_canal
GROUP BY ca.etapa_funil, ca.nome_canal, ca.tipo_canal
ORDER BY CASE ca.etapa_funil WHEN 'Autosserviço' THEN 1 WHEN 'Nível 1' THEN 2 WHEN 'Nível 2' THEN 3 ELSE 4 END, ca.nome_canal
""")

dados_vcredit.default.funil_atendimento_gold: 4 linhas


## Consulta | etapa_mensal

In [0]:
criar_gold(GOLD_ETAPA_MENSAL, f"""
SELECT
  c.ano_mes,
  ca.etapa_funil,
  COUNT(*)                                                    AS chamados,
  ROUND(100 * AVG(CAST(f.abandonado AS INT)), 2)              AS pct_abandono,
  ROUND(100 * AVG(CAST(f.resolvido AS INT)), 2)               AS pct_resolucao,
  ROUND(AVG(f.tempo_espera_seg) / 60, 2)                      AS tme_min,
  ROUND(AVG(f.tempo_atendimento_seg) / 60, 2)                 AS tma_min,
  ROUND(AVG(f.custo), 2)                                      AS custo_medio,
  ROUND(100 * AVG(CAST(f.satisfeito AS INT)), 2)              AS csat_pct
FROM {GOLD_FATO} f
JOIN {GOLD_DIM_CALENDARIO} c ON f.id_data = c.id_data
JOIN {GOLD_DIM_CANAL} ca ON f.id_canal = ca.id_canal
WHERE c.ano_mes IS NOT NULL
GROUP BY c.ano_mes, ca.etapa_funil
ORDER BY c.ano_mes, ca.etapa_funil
""")


dados_vcredit.default.etapa_mensal_gold: 57 linhas


## Consulta | motivos_pareto

In [0]:
criar_gold(GOLD_MOTIVOS_PARETO, f"""
WITH x AS (
  SELECT
    m.nome_motivo                                             AS motivo,
    m.categoria,
    m.criticidade,
    COUNT(*)                                                  AS chamados,
    COUNT_IF(ca.etapa_funil = 'Autosserviço')                 AS chamados_autosservico,
    COUNT_IF(ca.etapa_funil = 'Nível 1')                      AS chamados_nivel1,
    COUNT_IF(ca.etapa_funil = 'Nível 2')                      AS chamados_nivel2,
    ROUND(100 * AVG(CAST(f.resolvido AS INT)), 2)             AS pct_resolucao,
    ROUND(100 * AVG(CAST(f.reincidente_7d AS INT)), 2)        AS pct_reincidencia,
    ROUND(AVG(f.tempo_atendimento_seg) / 60, 2)               AS tma_min,
    ROUND(SUM(f.custo), 2)                                    AS custo_total,
    ROUND(AVG(f.custo), 2)                                    AS custo_medio,
    ROUND(100 * AVG(CAST(f.satisfeito AS INT)), 2)            AS csat_pct
  FROM {GOLD_FATO} f
  JOIN {GOLD_DIM_MOTIVO} m ON f.id_motivo = m.id_motivo
  JOIN {GOLD_DIM_CANAL} ca ON f.id_canal = ca.id_canal
  GROUP BY m.nome_motivo, m.categoria, m.criticidade
)
SELECT
  motivo, categoria, criticidade, chamados,
  ROUND(100 * chamados / SUM(chamados) OVER (), 2)            AS pct_chamados,
  ROUND(100 * SUM(chamados) OVER (ORDER BY chamados DESC, motivo
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(chamados) OVER (), 2) AS pct_acumulado,
  ROUND(100 * (chamados_nivel1 + chamados_nivel2) / chamados, 2) AS pct_chegou_humano,
  ROUND(100 * chamados_nivel2 / chamados, 2)                  AS pct_nivel2,
  pct_resolucao, pct_reincidencia, tma_min, custo_total, custo_medio, csat_pct,
  ROUND(100 * custo_total / SUM(custo_total) OVER (), 2)      AS pct_custo
FROM x
ORDER BY chamados DESC, motivo
""")

dados_vcredit.default.motivos_pareto_gold: 14 linhas


## Consulta | motivo_etapa

In [0]:
criar_gold(GOLD_MOTIVO_ETAPA, f"""
SELECT
  m.categoria,
  m.nome_motivo                                               AS motivo,
  ca.etapa_funil,
  COUNT(*)                                                    AS chamados,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY m.nome_motivo), 2) AS pct_do_motivo,
  ROUND(100 * AVG(CAST(f.resolvido AS INT)), 2)               AS pct_resolucao,
  ROUND(AVG(f.tempo_atendimento_seg) / 60, 2)                 AS tma_min,
  ROUND(AVG(f.custo), 2)                                      AS custo_medio
FROM {GOLD_FATO} f
JOIN {GOLD_DIM_MOTIVO} m ON f.id_motivo = m.id_motivo
JOIN {GOLD_DIM_CANAL} ca ON f.id_canal = ca.id_canal
GROUP BY m.categoria, m.nome_motivo, ca.etapa_funil
ORDER BY m.nome_motivo, ca.etapa_funil
""")

dados_vcredit.default.motivo_etapa_gold: 42 linhas


## Consulta | demanda_dia_hora

In [0]:
criar_gold(GOLD_DEMANDA_DIA_HORA, f"""
SELECT
  c.dia_semana,
  c.nome_dia,
  h.id_hora                                                   AS hora_abertura,
  h.faixa_horaria,
  COUNT(*)                                                    AS chamados,
  ROUND(100 * AVG(CAST(f.abandonado AS INT)), 2)              AS pct_abandono,
  ROUND(AVG(f.tempo_espera_seg) / 60, 2)                      AS tme_min,
  ROUND(percentile_approx(f.tempo_espera_seg, 0.9) / 60, 2)   AS p90_espera_min
FROM {GOLD_FATO} f
JOIN {GOLD_DIM_CALENDARIO} c ON f.id_data = c.id_data
JOIN {GOLD_DIM_HORA} h ON f.id_hora = h.id_hora
WHERE f.id_data <> -1 AND f.id_hora <> -1
GROUP BY c.dia_semana, c.nome_dia, h.id_hora, h.faixa_horaria
ORDER BY c.dia_semana, h.id_hora
""")

dados_vcredit.default.demanda_dia_hora_gold: 168 linhas


## Consulta | atendentes_desempenho

In [0]:
criar_gold(GOLD_ATENDENTES_DESEMPENHO, f"""
WITH x AS (
  SELECT
    ate.id_atendente, ate.rotulo_atendente, ate.nivel_atendimento, ate.data_admissao, ate.data_desligamento,
    COUNT(*)                                                  AS chamados,
    COUNT(f.tempo_atendimento_seg)                            AS atendimentos_validos,
    ROUND(AVG(f.tempo_atendimento_seg) / 60, 2)               AS tma_min,
    ROUND(percentile_approx(f.tempo_atendimento_seg, 0.9) / 60, 2) AS p90_tma_min,
    ROUND(100 * AVG(CAST(f.resolvido AS INT)), 2)             AS pct_resolucao,
    ROUND(100 * AVG(CAST(f.satisfeito AS INT)), 2)            AS csat_pct,
    ROUND(AVG(f.custo), 2)                                    AS custo_medio,
    MIN(c.data)                                               AS primeiro_chamado,
    MAX(c.data)                                               AS ultimo_chamado,
    COUNT_IF(q.atendente_fora_vigencia)                       AS chamados_fora_vigencia
  FROM {GOLD_FATO} f
  JOIN {GOLD_DIM_ATENDENTE} ate ON f.id_atendente = ate.id_atendente
  JOIN {GOLD_DIM_CALENDARIO} c ON f.id_data = c.id_data
  JOIN {GOLD_QUALIDADE_CHAMADO} q ON f.id_chamado = q.id_chamado
  WHERE f.id_atendente > 0
  GROUP BY ate.id_atendente, ate.rotulo_atendente, ate.nivel_atendimento, ate.data_admissao, ate.data_desligamento
)
SELECT
  x.*,
  chamados >= {AMOSTRA_MIN_ATENDENTE}                         AS amostra_suficiente,
  ROUND(MONTHS_BETWEEN(LEAST(ultimo_chamado, COALESCE(data_desligamento, ultimo_chamado)), data_admissao), 1) AS tempo_casa_meses_no_ultimo_chamado,
  ROUND(pct_resolucao - AVG(CASE WHEN chamados >= {AMOSTRA_MIN_ATENDENTE} THEN pct_resolucao END)
        OVER (PARTITION BY nivel_atendimento), 2)             AS pct_resolucao_vs_media_nivel,
  ROUND(tma_min - AVG(CASE WHEN chamados >= {AMOSTRA_MIN_ATENDENTE} THEN tma_min END)
        OVER (PARTITION BY nivel_atendimento), 2)             AS tma_vs_media_nivel,
  RANK() OVER (PARTITION BY nivel_atendimento, chamados >= {AMOSTRA_MIN_ATENDENTE}
               ORDER BY pct_resolucao DESC)                   AS rank_resolucao_no_nivel
FROM x
ORDER BY nivel_atendimento, chamados DESC
""")

dados_vcredit.default.atendentes_desempenho_gold: 25 linhas


## Consulta | satisfacao_drivers

In [0]:
_dims = [
    ("faixa_espera", "faixa_espera"),
    ("etapa_funil", "etapa_funil"),
    ("canal", "canal"),
    ("categoria_motivo", "categoria_motivo"),
    ("resolucao", "situacao_resolucao"),
    ("reincidencia", "reincidencia"),
    ("faixa_horaria", "faixa_horaria"),
]
_union_dims = "\nUNION ALL\n".join(
    f"""SELECT '{nome}' AS dimensao, {expr} AS valor, COUNT(*) AS respostas,
       ROUND(100 * AVG(CAST(satisfeito AS INT)), 2) AS csat_pct,
       ROUND(AVG(nota), 2) AS nota_media
FROM r GROUP BY 2"""
    for nome, expr in _dims
)
 
criar_gold(GOLD_SATISFACAO_DRIVERS, f"""
WITH r AS (
  SELECT
    f.nota, f.satisfeito,
    fe.rotulo_faixa                                           AS faixa_espera,
    ca.etapa_funil,
    ca.nome_canal                                             AS canal,
    m.categoria                                               AS categoria_motivo,
    h.faixa_horaria,
    CASE WHEN f.resolvido THEN 'Resolvido' WHEN f.resolvido = false THEN 'Não resolvido' ELSE 'Sem informação' END AS situacao_resolucao,
    CASE WHEN f.reincidente_7d THEN 'Reincidente (7d)' ELSE 'Primeiro contato' END AS reincidencia
  FROM {GOLD_FATO} f
  JOIN {GOLD_DIM_FAIXA_ESPERA} fe ON f.id_faixa_espera = fe.id_faixa_espera
  JOIN {GOLD_DIM_CANAL} ca ON f.id_canal = ca.id_canal
  JOIN {GOLD_DIM_MOTIVO} m ON f.id_motivo = m.id_motivo
  JOIN {GOLD_DIM_HORA} h ON f.id_hora = h.id_hora
  WHERE f.nota IS NOT NULL
)
SELECT * FROM (
{_union_dims}
)
ORDER BY dimensao, valor
""")

dados_vcredit.default.satisfacao_drivers_gold: 28 linhas


## Consulta | perfil_cliente

In [0]:
if DIM_CLIENTE_CRIADA:
    _dims_cli = ["faixa_etaria", "regiao"]
    _union_cli = "\nUNION ALL\n".join(
        f"""SELECT '{nome}' AS dimensao, {nome} AS valor, etapa_funil,
       COUNT(*) AS chamados,
       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY {nome}), 2) AS pct_da_dimensao,
       ROUND(100 * AVG(CAST(resolvido AS INT)), 2) AS pct_resolucao,
       ROUND(100 * AVG(CAST(abandonado AS INT)), 2) AS pct_abandono,
       ROUND(100 * AVG(CAST(satisfeito AS INT)), 2) AS csat_pct,
       ROUND(AVG(custo), 2) AS custo_medio
FROM p GROUP BY {nome}, etapa_funil"""
        for nome in _dims_cli
    )
    criar_gold(GOLD_PERFIL_CLIENTE, f"""
    WITH p AS (
      SELECT f.resolvido, f.abandonado, f.satisfeito, f.custo, ca.etapa_funil, cl.faixa_etaria, cl.regiao
      FROM {GOLD_FATO} f
      JOIN {GOLD_DIM_CANAL} ca ON f.id_canal = ca.id_canal
      JOIN {GOLD_DIM_CLIENTE} cl ON f.id_cliente_hash = cl.id_cliente_hash
    )
    SELECT * FROM (
    {_union_cli}
    )
    ORDER BY dimensao, valor, etapa_funil
    """)
else:
    spark.sql(f"DROP TABLE IF EXISTS {GOLD_PERFIL_CLIENTE}")
    print("perfil_cliente_gold não criada: depende da dim_cliente_gold.")

dados_vcredit.default.perfil_cliente_gold: 36 linhas


## Consulta | qualidade_dados

In [0]:
_DQ = [
    ("Tempos",    "chamado_sem_registro_na_base_hora",           "NOT q.tem_registro_hora"),
    ("Tempos",    "erro_parse_abertura",                         "q.abertura_erro_parse"),
    ("Tempos",    "erro_parse_inicio",                           "q.inicio_erro_parse"),
    ("Tempos",    "erro_parse_fim",                              "q.fim_erro_parse"),
    ("Tempos",    "ano_de_abertura_corrigido",                   "q.ano_abertura_corrigido"),
    ("Tempos",    "virada_de_dia_aplicada",                      "q.dia_fim_corrigido"),
    ("Tempos",    "atendido_sem_espera_valida_apos_tratamento",  "f.inicio_ts IS NOT NULL AND f.tempo_espera_seg IS NULL"),
    ("Tempos",    "atendido_sem_duracao_valida_apos_tratamento", "f.inicio_ts IS NOT NULL AND f.tempo_atendimento_seg IS NULL"),
    ("Tempos",    "abandonado_sem_inicio_(informativo)",         "f.abandonado"),
    ("Canal",     "canal_nao_mapeado",                           "NOT q.canal_mapeado"),
    ("Motivo",    "motivo_nao_mapeado",                          "NOT q.motivo_mapeado"),
    ("Resolução", "resolvido_nao_informado",                     "q.dq_resolvido_ausente"),
    ("Resolução", "resolvido_com_valor_invalido",                "q.dq_resolvido_invalido AND NOT q.dq_resolvido_ausente"),
    ("Custo",     "chamado_sem_custo",                           "q.qtd_registros_custo = 0"),
    ("Custo",     "custo_invalido_ou_nao_positivo",              "q.custo_com_problema"),
    ("Custo",     "chamado_com_mais_de_um_custo",                "q.qtd_registros_custo > 1"),
    ("Pesquisa",  "chamado_sem_pesquisa",                        "NOT q.tem_pesquisa"),
    ("Pesquisa",  "nota_fora_de_1_a_5",                          "q.dq_nota_invalida"),
    ("Pesquisa",  "nota_recuperada_pelo_ajuste",                 "f.nota IS NULL AND f.nota_ajustada IS NOT NULL"),
    ("Atendente", "atendente_ausente_na_origem_(informativo)",   "q.status_atendente = 'ausente_na_origem'"),
    ("Atendente", "atendente_invalido_na_origem",                "q.status_atendente = 'invalido_na_origem'"),
    ("Atendente", "chamado_humano_sem_atendente_valido",         "f.id_atendente < 0 AND ca.etapa_funil IN ('Nível 1', 'Nível 2')"),
    ("Atendente", "atendente_nao_encontrado_no_cadastro",        "q.atendente_nao_encontrado"),
    ("Atendente", "chamado_fora_da_vigencia_do_atendente",       "q.atendente_fora_vigencia"),
    ("Atendente", "autosservico_com_atendente",                  "q.autosservico_com_atendente"),
    ("Atendente", "nivel_do_atendente_diverge_do_canal",         "q.atendente_nivel_diverge_canal"),
    ("Cliente",   "chamado_sem_id_cliente",                      "q.sem_id_cliente"),
]
_agg = ",\n    ".join(f"COUNT_IF({expr}) AS c{i}" for i, (_, _, expr) in enumerate(_DQ))
_stack = ", ".join(f"'{g}', '{n}', c{i}" for i, (g, n, _) in enumerate(_DQ))
 
criar_gold(GOLD_QUALIDADE_DADOS, f"""
WITH agg AS (
  SELECT COUNT(*) AS total,
    {_agg}
  FROM {GOLD_FATO} f
  JOIN {GOLD_QUALIDADE_CHAMADO} q ON f.id_chamado = q.id_chamado
  JOIN {GOLD_DIM_CANAL} ca ON f.id_canal = ca.id_canal
)
SELECT grupo, indicador, qtd, ROUND(100 * qtd / total, 2) AS pct_chamados
FROM (
  SELECT total, stack({len(_DQ)}, {_stack}) AS (grupo, indicador, qtd)
  FROM agg
)
ORDER BY grupo, qtd DESC
""")

dados_vcredit.default.qualidade_dados_gold: 27 linhas


## Consulta | impacto_indicadores

In [0]:
_FQ = f"FROM {GOLD_FATO} f JOIN {GOLD_QUALIDADE_CHAMADO} q ON f.id_chamado = q.id_chamado"
 
criar_gold(GOLD_IMPACTO_INDICADORES, f"""
SELECT *, ROUND(valor_b - valor_a, 2) AS diferenca FROM (
  SELECT 'TME (min)' AS indicador,
    'bruto: todas as diferenças, inclusive negativas' AS metodo_a,
    ROUND(AVG(timestampdiff(SECOND, q.abertura_original_ts, f.inicio_ts)) / 60, 2) AS valor_a,
    COUNT(timestampdiff(SECOND, q.abertura_original_ts, f.inicio_ts)) AS n_a,
    'tratado: sequência válida, com correções' AS metodo_b,
    ROUND(AVG(f.tempo_espera_seg) / 60, 2) AS valor_b,
    COUNT(f.tempo_espera_seg) AS n_b
  {_FQ}
  UNION ALL
  SELECT 'TME (min)',
    'só válidos, sem correção de ano/dia',
    ROUND(AVG(CASE WHEN f.inicio_ts >= q.abertura_original_ts THEN timestampdiff(SECOND, q.abertura_original_ts, f.inicio_ts) END) / 60, 2),
    COUNT_IF(f.inicio_ts >= q.abertura_original_ts),
    'tratado: sequência válida, com correções',
    ROUND(AVG(f.tempo_espera_seg) / 60, 2),
    COUNT(f.tempo_espera_seg)
  {_FQ}
  UNION ALL
  SELECT 'TMA (min)',
    'bruto: todas as diferenças, inclusive negativas',
    ROUND(AVG(timestampdiff(SECOND, f.inicio_ts, q.fim_original_ts)) / 60, 2),
    COUNT(timestampdiff(SECOND, f.inicio_ts, q.fim_original_ts)),
    'tratado: sequência válida, com correções',
    ROUND(AVG(f.tempo_atendimento_seg) / 60, 2),
    COUNT(f.tempo_atendimento_seg)
  {_FQ}
  UNION ALL
  SELECT 'TMA (min)',
    'só válidos, sem correção de ano/dia',
    ROUND(AVG(CASE WHEN f.inicio_ts >= q.abertura_original_ts AND q.fim_original_ts >= f.inicio_ts
                   THEN timestampdiff(SECOND, f.inicio_ts, q.fim_original_ts) END) / 60, 2),
    COUNT_IF(f.inicio_ts >= q.abertura_original_ts AND q.fim_original_ts >= f.inicio_ts),
    'tratado: sequência válida, com correções',
    ROUND(AVG(f.tempo_atendimento_seg) / 60, 2),
    COUNT(f.tempo_atendimento_seg)
  {_FQ}
  UNION ALL
  SELECT 'CSAT (%)',
    'nota válida (1 a 5)',
    ROUND(100 * AVG(CAST(f.satisfeito AS INT)), 2), COUNT(f.nota),
    'nota ajustada (6→5, 0/-1→1)',
    ROUND(100 * AVG(CAST(f.satisfeito_ajustado AS INT)), 2), COUNT(f.nota_ajustada)
  {_FQ}
  UNION ALL
  SELECT 'Custo médio por chamado (R$)',
    'só chamados com custo válido',
    ROUND(AVG(f.custo), 2), COUNT(f.custo),
    'diluído em todos os chamados (sem custo = 0)',
    ROUND(SUM(f.custo) / COUNT(*), 2), COUNT(*)
  {_FQ}
  UNION ALL
  SELECT 'Taxa de resolução (%)',
    'sobre todos os chamados',
    ROUND(100 * AVG(CAST(f.resolvido AS INT)), 2), COUNT(f.resolvido),
    'só chamados atendidos (exclui abandonados)',
    ROUND(100 * AVG(CASE WHEN NOT f.abandonado THEN CAST(f.resolvido AS INT) END), 2),
    COUNT_IF(NOT f.abandonado AND f.resolvido IS NOT NULL)
  {_FQ}
)
""")

dados_vcredit.default.impacto_indicadores_gold: 7 linhas


## Consulta | simulacao_ganhos

In [0]:
criar_gold(GOLD_SIMULACAO_GANHOS, f"""
WITH meses AS (
  SELECT COUNT(DISTINCT c.ano_mes) AS n
  FROM {GOLD_FATO} f
  JOIN {GOLD_DIM_CALENDARIO} c ON f.id_data = c.id_data
),
auto_global AS (
  SELECT AVG(f.custo) AS custo_medio
  FROM {GOLD_FATO} f
  JOIN {GOLD_DIM_CANAL} ca ON f.id_canal = ca.id_canal
  WHERE ca.etapa_funil = 'Autosserviço'
),
a AS (
  SELECT 'A) Deflexão de baixa criticidade para autosserviço' AS cenario,
         m.nome_motivo AS item,
         COUNT_IF(ca.etapa_funil IN ('Nível 1', 'Nível 2')) AS chamados_base,
         CAST({PCT_DEFLEXAO} AS DOUBLE) AS premissa_pct,
         AVG(CASE WHEN ca.etapa_funil IN ('Nível 1', 'Nível 2') THEN f.custo END)
           - COALESCE(AVG(CASE WHEN ca.etapa_funil = 'Autosserviço' THEN f.custo END), MAX(g.custo_medio)) AS ganho_unitario
  FROM {GOLD_FATO} f
  JOIN {GOLD_DIM_MOTIVO} m ON f.id_motivo = m.id_motivo
  JOIN {GOLD_DIM_CANAL} ca ON f.id_canal = ca.id_canal
  CROSS JOIN auto_global g
  WHERE m.criticidade = 'baixa' AND m.id_motivo > 0
  GROUP BY m.nome_motivo
),
b AS (
  SELECT 'B) Redução de reincidência (recontato do mesmo motivo)' AS cenario,
         m.nome_motivo AS item,
         COUNT_IF(f.reincidente_7d) AS chamados_base,
         CAST({PCT_REDUCAO_REINCIDENCIA} AS DOUBLE) AS premissa_pct,
         AVG(CASE WHEN f.reincidente_7d THEN f.custo END) AS ganho_unitario
  FROM {GOLD_FATO} f
  JOIN {GOLD_DIM_MOTIVO} m ON f.id_motivo = m.id_motivo
  GROUP BY m.nome_motivo
),
calc AS (
  SELECT cenario, item, chamados_base, premissa_pct,
         chamados_base * premissa_pct AS chamados_evitados,
         ganho_unitario,
         chamados_base * premissa_pct * ganho_unitario AS economia_periodo
  FROM (SELECT * FROM a UNION ALL SELECT * FROM b)
  WHERE chamados_base > 0
)
SELECT * FROM (
  SELECT cenario, item, chamados_base, premissa_pct,
         ROUND(chamados_evitados, 0) AS chamados_evitados,
         ROUND(ganho_unitario, 2) AS ganho_unitario,
         ROUND(economia_periodo, 2) AS economia_periodo,
         ROUND(economia_periodo / m.n, 2) AS economia_mensal_media,
         ROUND(economia_periodo / m.n * 12, 2) AS economia_anualizada
  FROM calc CROSS JOIN meses m
  UNION ALL
  SELECT cenario, '** TOTAL **', SUM(chamados_base), MAX(premissa_pct),
         ROUND(SUM(chamados_evitados), 0), CAST(NULL AS DOUBLE),
         ROUND(SUM(economia_periodo), 2),
         ROUND(SUM(economia_periodo) / m.n, 2),
         ROUND(SUM(economia_periodo) / m.n * 12, 2)
  FROM calc CROSS JOIN meses m
  GROUP BY cenario, m.n
)
ORDER BY cenario, CASE WHEN item = '** TOTAL **' THEN 0 ELSE 1 END, economia_periodo DESC
""")

dados_vcredit.default.simulacao_ganhos_gold: 20 linhas


## Verificação | Resumo das tabelas Gold

In [0]:
_tabelas = [
    GOLD_DIM_CALENDARIO, GOLD_DIM_HORA, GOLD_DIM_FAIXA_ESPERA, GOLD_DIM_CANAL, GOLD_DIM_MOTIVO,
    GOLD_DIM_ATENDENTE, GOLD_DIM_CLIENTE, GOLD_FATO, GOLD_QUALIDADE_CHAMADO,
    GOLD_VISAO_GERAL, GOLD_KPI_MENSAL, GOLD_COMPARATIVO_YOY, GOLD_FUNIL, GOLD_ETAPA_MENSAL,
    GOLD_MOTIVOS_PARETO, GOLD_MOTIVO_ETAPA, GOLD_DEMANDA_DIA_HORA, GOLD_ATENDENTES_DESEMPENHO,
    GOLD_SATISFACAO_DRIVERS, GOLD_PERFIL_CLIENTE, GOLD_QUALIDADE_DADOS, GOLD_IMPACTO_INDICADORES,
    GOLD_SIMULACAO_GANHOS,
]
for _n in _tabelas:
    try:
        print(f"{_n}: {spark.table(_n).count():,} linhas")
    except Exception:
        print(f"{_n}: não criada")

dados_vcredit.default.dim_calendario_gold: 731 linhas
dados_vcredit.default.dim_hora_gold: 25 linhas
dados_vcredit.default.dim_faixa_espera_gold: 6 linhas
dados_vcredit.default.dim_canal_gold: 5 linhas
dados_vcredit.default.dim_motivo_gold: 15 linhas
dados_vcredit.default.dim_atendente_gold: 28 linhas
dados_vcredit.default.dim_cliente_gold: 47,705 linhas
dados_vcredit.default.fato_atendimento_gold: 131,838 linhas
dados_vcredit.default.qualidade_chamado_gold: 131,838 linhas
dados_vcredit.default.visao_geral_gold: 1 linhas
dados_vcredit.default.kpi_mensal_gold: 19 linhas
dados_vcredit.default.comparativo_yoy_gold: 12 linhas
dados_vcredit.default.funil_atendimento_gold: 4 linhas
dados_vcredit.default.etapa_mensal_gold: 57 linhas
dados_vcredit.default.motivos_pareto_gold: 14 linhas
dados_vcredit.default.motivo_etapa_gold: 42 linhas
dados_vcredit.default.demanda_dia_hora_gold: 168 linhas
dados_vcredit.default.atendentes_desempenho_gold: 25 linhas
dados_vcredit.default.satisfacao_drivers_gol